In [149]:
from dotenv import load_dotenv
import os
import requests
from pprint import pprint
import pandas as pd
import missingno as mno
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import time
import missingno as mno

load_dotenv()

True

In [6]:
from dotenv import load_dotenv
import os

result = load_dotenv(dotenv_path=".ski.env")
print("Loaded successfully:", result)

SKIDDLE_API_KEY = os.getenv("SKIDDLE_API_KEY")
print("Key found:", key is not None)

Loaded successfully: True
Key found: True


In [63]:
import requests

response = requests.get(
    "https://www.skiddle.com/api/v1/events/search/",
    params={
        "api_key": SKIDDLE_API_KEY,
        "country": "GB",
        "eventcode": "FEST",       # filters to Festivals specifically — genuinely useful for you
        "description": 1,          # includes artist/genre info in the response
        "minDate": "2026-06-01",
        "maxDate": "2026-08-31",
        "limit": 100,              # max per request
        "offset": 0
    }
)
print(response.status_code)
print(response.url)
data = response.json()

200
https://www.skiddle.com/api/v1/events/search/?api_key=29189e75d8a1fa765c8b04f57d254d86&country=GB&eventcode=FEST&description=1&minDate=2026-06-01&maxDate=2026-08-31&limit=100&offset=0


In [67]:
pprint(data.keys()) 

dict_keys(['error', 'totalcount', 'pagecount', 'results', 'elastic', 'requestId'])


In [65]:
pprint(data)

{'elastic': {'timing': 151},
 'error': 0,
 'pagecount': 100,
 'requestId': 'api_6a50b86b669e88.63456994',
 'results': [{'EventCode': 'FEST',
              'artists': [{'artistid': '123468608',
                           'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/8/123468608_1_1024.jpg',
                           'name': 'Shy FX',
                           'spotifyartisturl': 'spotify:artist:5oDtp2FC8VqBjTx1aT4P5j'},
                          {'artistid': '123483430',
                           'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/0/123483430_1_1024.jpg',
                           'name': 'Pendulum',
                           'spotifyartisturl': 'spotify:artist:7MqnCTCAX6SsIYYdJCQj9B'},
                          {'artistid': '123594984',
                           'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/4/123594984_1_1024.jpg',
                           'name': 'HOLY PRIEST',
                           'spotifyartisturl': 'spotify:artist:5UG2ipdnA4vk8Eevkf1s

The only data we are interested in is the results so lets overwrite data

In [66]:
pprint(type(data))

<class 'dict'>


Lets paginate

In [69]:
total = data.get('totalcount')
pagecount = data.get('pagecount')
print(f"Total results: {total}")
print(f"Page count: {pagecount}")
print(data.get('error'))

Total results: 1057
Page count: 100
0


In [71]:
def fetch_skiddle_page(offset):
    response = requests.get(
        "https://www.skiddle.com/api/v1/events/search/",
        params={
            "api_key": SKIDDLE_API_KEY,
            "country": "GB",
            "eventcode": "FEST",
            "description": 1,
            "minDate": "2026-06-01",
            "maxDate": "2026-08-31",
            "limit": 100,
            "offset": offset
        }
    )
    response.raise_for_status()
    return response.json()

first_page = fetch_skiddle_page(0)
total = int(first_page.get('totalcount'))  # Convert string to integer
print(f"Total results: {total}")

all_skiddle_events = list(first_page['results'])

offset = 100
while offset < total:  # Now both offset and total are integers
    time.sleep(0.25)
    page = fetch_skiddle_page(offset)
    all_skiddle_events.extend(page['results'])
    print(f"Fetched offset {offset}, running total: {len(all_skiddle_events)}")
    offset += 100

print(f"\nFinal total collected: {len(all_skiddle_events)}")

Total results: 1056
Fetched offset 100, running total: 200
Fetched offset 200, running total: 300
Fetched offset 300, running total: 400
Fetched offset 400, running total: 500
Fetched offset 500, running total: 600
Fetched offset 600, running total: 700
Fetched offset 700, running total: 800
Fetched offset 800, running total: 900
Fetched offset 900, running total: 1000
Fetched offset 1000, running total: 1056

Final total collected: 1056


In [72]:
ids = [e.get('EventID') or e.get('id') for e in all_skiddle_events]  # confirm actual id field name once you see one event's keys
print(f"Total events: {len(ids)}")
print(f"Unique IDs: {len(set(ids))}")

Total events: 1056
Unique IDs: 1056


In [73]:
df = pd.json_normalize(all_skiddle_events)
print(df.shape)
df.columns.tolist()

(1056, 64)


['id',
 'listingid',
 'isSBT',
 'uniquelistingidentifier',
 'hascollapsedresults',
 'countcollapsedresults',
 'EventCode',
 'eventname',
 'cancelled',
 'cancellationDate',
 'cancellationType',
 'cancellationReason',
 'rescheduledDate',
 'imageurl',
 'largeimageurl',
 'xlargeimageurl',
 'xlargeimageurlWebP',
 'link',
 'date',
 'startdate',
 'enddate',
 'description',
 'minage',
 'imgoing',
 'goingtos',
 'goingtocount',
 'tickets',
 'entryprice',
 'eventvisibility',
 'ticketUrl',
 'hotSeller',
 'festivalId',
 'headerHex',
 'currency',
 'artists',
 'genres',
 'healthAndSafety',
 'venue.id',
 'venue.name',
 'venue.address',
 'venue.town',
 'venue.postcode_lookup',
 'venue.postcode',
 'venue.region',
 'venue.country',
 'venue.phone',
 'venue.latitude',
 'venue.longitude',
 'venue.type',
 'venue.rating',
 'venue.reviewCount',
 'openingtimes.doorsopen',
 'openingtimes.doorsclose',
 'openingtimes.lastentry',
 'ticketpricing.minPrice',
 'ticketpricing.maxPrice',
 'festival.id',
 'festival.name'

In [78]:
df.head()

,id,listingid,isSBT,uniquelistingidentifier,hascollapsedresults,countcollapsedresults,EventCode,eventname,cancelled,cancellationDate,...,ticketpricing.minPrice,ticketpricing.maxPrice,festival.id,festival.name,festival.earliestStartDate,rep.enabled,rep.eligible,rep.minCommission,rep.maxCommission,ticketpricing
0,41585614,2147482,False,2147482,False,0,FEST,Forbidden Forest 2026,0,,...,150.0,150.0,1196.0,Forbidden Forest Festival,NaN,False,NaN,NaN,NaN,NaN
1,41643401,2161007,False,2161007,False,0,FEST,Bulletproof Festival 2026,0,,...,0.0,0.0,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
2,41392590,2116445,False,2116445,False,0,FEST,Symphonic Ibiza - At The Beach,0,,...,20.0,750.0,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
3,41997035,2198751,False,2198751,False,0,FEST,AnExperience Festival,0,,...,0.0,0.0,3001.0,AnExperience Festival,NaN,False,NaN,NaN,NaN,NaN
4,40660839,1949915,False,1949915,False,0,FEST,Fields of Éire- Irish Music Festival Liverpool...,0,,...,0.0,0.0,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN


In [100]:
df['id'].nunique()

1056

In [96]:
nested_cols = []

for col in df.columns:
    non_null = df[col].dropna()
    if len(non_null) == 0:
        continue
    sample = non_null.iloc[0]
    if isinstance(sample, (list, dict)):
        nested_cols.append(col)

print(nested_cols)

['artists', 'genres']


In [98]:
df['artists'].iloc[3]

[{'artistid': '123562232',
  'name': 'BCUC',
  'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/2/123562232_1_1024.jpg',
  'spotifyartisturl': 'spotify:artist:5CGnQOjeOoZW4a4FoPhUxW'},
 {'artistid': '123459839',
  'name': 'Dennis Bovell',
  'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/9/123459839_267.jpg',
  'spotifyartisturl': 'spotify:artist:0xJuAKhVgEfuiEXjyLEuC6'},
 {'artistid': '123505414',
  'name': 'Tim Edey',
  'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/4/123505414_267.jpg',
  'spotifyartisturl': 'spotify:artist:53SSTqCzcKtd3uKl29Krc1'},
 {'artistid': '123512720',
  'name': 'Equinox',
  'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/0/123512720_1_267.jpg',
  'spotifyartisturl': 'spotify:artist:3TGa5U9uBpIQXoI2emyXMo'},
 {'artistid': '123518520',
  'name': 'Jerome Hill',
  'image': 'https://d1mdxzfl9p8pzo.cloudfront.net/0/123518520_2_1024.jpg',
  'spotifyartisturl': 'spotify:artist:4K6SrfGnkgHak2f007UkvB'},
 {'artistid': '123527580',
  'name': 'The Cheeky Girls',
  'imag

In [95]:
df = df.drop(columns=['healthAndSafety'])

Building 2 tables for our nested columns: df_artists & df_genres

In [99]:
def build_nested_table(df, id_col, nested_col):
    """
    Turns a nested list-of-dicts (or single dict) column into its own table,
    one row per item, linked back to the parent via id_col.
    Unwraps ONE level of nested dicts (e.g. genre: {id, name} -> genre_id, genre_name).
    Lists nested inside items are skipped (too complex for a flat row).
    """
    rows = []
    for _, row in df.iterrows():
        parent_id = row[id_col]
        value = row.get(nested_col)
        if value is None:
            continue
        items = value if isinstance(value, list) else [value]
        for item in items:
            if not isinstance(item, dict):
                continue
            flat_item = {f"event_{id_col}": parent_id}   # renamed to avoid collision
            for key, val in item.items():
                if isinstance(val, dict):
                    for subkey, subval in val.items():
                        flat_item[f"{key}_{subkey}"] = subval
                elif isinstance(val, list):
                    continue
                else:
                    flat_item[key] = val   # venue's own 'id' now safely separate
            rows.append(flat_item)
    return pd.DataFrame(rows)

In [103]:
df_artists = build_nested_table(df, id_col='id', nested_col='artists')
df_genres = build_nested_table(df, id_col='id', nested_col='genres')

print("artists:", df_artists.shape, df_artists.columns.tolist())
print("genres:", df_genres.shape, df_genres.columns.tolist())

artists: (7552, 5) ['event_id', 'artistid', 'name', 'image', 'spotifyartisturl']
genres: (2796, 3) ['event_id', 'genreid', 'name']


In [114]:
df_genres.shape

(2796, 3)

In [115]:
df_skiddle = df.copy()

Creating df_skiddle for manipulation

In [116]:
df_skiddle = df_skiddle.drop(columns=['artists'])
df_skiddle = df_skiddle.drop(columns=['genres'])

In [118]:
df_skiddle.shape

(1056, 61)

In [139]:

# print("skiddle_data:", df_skiddle.shape, df_skiddle.columns.tolist())
# print("artists:", df_artists.shape, df_artists.columns.tolist())
print("genres:", df_genres.shape, df_genres.columns.tolist())

genres: (2796, 3) ['event_id', 'genreid', 'name']


In [141]:
df_genres.head()

,event_id,genreid,name
0,41585614,1,House
1,41585614,8,Drum and Bass
2,41585614,9,Techno
3,41585614,14,Tech House
4,41997035,24,Dancehall


Lets build our df_skiddle_genres table

In [147]:
skiddle_cols = ['id', 'eventname', 'startdate', 'enddate', 'venue.town', 'venue.region', 'cancelled']
df_skiddle_genre = df_skiddle[skiddle_cols].copy()

genre_summary = df_genres.rename(columns={'name': 'genre_name'}).groupby('event_id').agg(
    genre_count=('genre_name', 'nunique'),
    genres=('genre_name', lambda x: list(x.unique()))
).reset_index()

artist_summary = df_artists.groupby('event_id').agg(
    artist_count=('name', 'count')
).reset_index()

df_skiddle_genre = df_skiddle_genre.merge(genre_summary, left_on='id', right_on='event_id', how='left')
df_skiddle_genre = df_skiddle_genre.merge(artist_summary, left_on='id', right_on='event_id', how='left', suffixes=('', '_artist'))

df_skiddle_genre = df_skiddle_genre.drop(columns=['event_id', 'event_id_artist'], errors='ignore')
df_skiddle_genre.head()

,id,eventname,startdate,enddate,venue.town,venue.region,cancelled,genre_count,genres,artist_count
0,41585614,Forbidden Forest 2026,2026-06-04T12:00:00+00:00,2026-06-07T23:00:00+00:00,Grantham,Nottingham,0,4.0,"[House, Drum and Bass, Techno, Tech House]",50.0
1,41643401,Bulletproof Festival 2026,2026-06-04T17:30:00+00:00,2026-06-06T22:30:00+00:00,London,London,0,NaN,NaN,26.0
2,41392590,Symphonic Ibiza - At The Beach,2026-06-05T15:00:00+00:00,2026-06-05T22:30:00+00:00,Weston Super-Mare,Bristol,0,NaN,NaN,2.0
3,41997035,AnExperience Festival,2026-06-05T14:00:00+00:00,2026-06-08T00:00:00+00:00,Huntingdon,Cambridgeshire,0,5.0,"[Dancehall, Ska, Salsa, World Music, Jungle]",16.0
4,40660839,Fields of Éire- Irish Music Festival Liverpool...,2026-06-05T16:30:00+00:00,2026-06-06T23:00:00+00:00,Liverpool,Merseyside,0,3.0,"[Acoustic, Folk, Country/Americana]",NaN


In [148]:
df_skiddle_genre.to_csv("df_skiddle_genre.csv", index=False)

Exploring whether we should drop the artist_count column.
artist_count has approximately 41% of its data filled with nulls. 

In [152]:
def null_vals(dataframe):
    """
    Show both number of nulls and the percentage of nulls in the whole column across a Pandas dataframe.
    """
    null_vals = dataframe.isnull().sum() # How many nulls in each column
    total_cnt = len(dataframe) # Total entries in the dataframe
    null_vals = pd.DataFrame(null_vals,columns=['null']) # Put the number of nulls in a single dataframe
    null_vals['percent'] = round((null_vals['null']/total_cnt)*100,3) # Round how many nulls are there, as %, of the df
    
    return null_vals.sort_values('percent', ascending=False) # Ordered from MOST to LEAST nulls

In [153]:
null_vals(df_skiddle_genre)

,null,percent
artist_count,436,41.288
genre_count,227,21.496
genres,227,21.496
id,0,0.000
eventname,0,0.000
startdate,0,0.000
enddate,0,0.000
venue.town,0,0.000
venue.region,0,0.000
cancelled,0,0.000


In [163]:
# Here, i am theorising whether the nulls are correlated with dates in the future - e.g to capture scenarios where the lineup of artists has not been announced. There is barely any variation in the mean dates hence
# showing that there is no correlation
null_dates = pd.to_datetime(df_skiddle_genre[df_skiddle_genre['artist_count'].isnull()]['startdate'])
non_null_dates = pd.to_datetime(df_skiddle_genre[df_skiddle_genre['artist_count'].notna()]['startdate'])

print("Null artist_count — date summary:")
print(null_dates.describe())
print("\nNon-null artist_count — date summary:")
print(non_null_dates.describe())

Null artist_count — date summary:
count                                    436
mean     2026-07-20 11:43:31.238531840+00:00
min                2026-06-05 10:15:00+00:00
25%                2026-06-27 12:00:00+00:00
50%                2026-07-25 02:45:00+00:00
75%                2026-08-09 12:15:00+00:00
max                2026-08-31 13:00:00+00:00
Name: startdate, dtype: object

Non-null artist_count — date summary:
count                                    620
mean     2026-07-21 00:47:51.290322432+00:00
min                2026-06-04 12:00:00+00:00
25%                2026-06-28 14:45:00+00:00
50%                2026-07-24 11:00:00+00:00
75%                2026-08-08 17:30:00+00:00
max                2026-08-31 12:00:00+00:00
Name: startdate, dtype: object


In [226]:
# Here, i am checking whether there is a correlation between the genres and the number of artist_count that is missing. 
# Only 5 categories (mix of genres or standalone) have a higher freq with 100 of the others being distinct genre categories. Hence no correlation again.

non_music_check = df_skiddle_genre[df_skiddle_genre['artist_count'].isnull()]['genres'].value_counts()
print(non_music_check)

genres
[Country/Americana]                                25
[R&B]                                              22
[R&B, Hip Hop, 1990s, 2000s]                       10
[House, Tech House, Pop, Indie, Rap]                8
[Rock, Pop, Indie, Indie Pop, Pop Rock]             6
                                                   ..
[Dancehall]                                         1
[Pop, Acoustic, Country/Americana]                  1
[House, Tech House, Disco House, Soulful House]     1
[House, Tech House, Funky House]                    1
[UK Garage, Drum and Bass]                          1
Name: count, Length: 150, dtype: int64


I decided to keep the artists_counts as null. This is because i plan to join this with the df_genre.csv that i created using the ticketmaster API. There might be cases where we drop duplicate fields so would like to keep this or re-evalute the nulls in the final combined table